# figureflow examples

Every example from `examples/` in one runnable notebook. Run a section's cell and the interactive widget renders inline (the canvas only appears in a notebook host: JupyterLab, Notebook, Colab, VS Code, marimo).

**Canvas controls:** left-drag box-selects nodes; middle/right-drag pans; Shift+click adds to the selection; Ctrl/Cmd+Z / +Shift+Z undo/redo; Ctrl/Cmd+C / +V copy/paste.

## Setup

```bash
pip install figureflow   # or, from a checkout: pip install -e .
```

In [ ]:
from figureflow import Edge, Flow, Node, Shape

## 01 · Quickstart

All 8 node shapes, the L1 `svg_path` and L2 `html` escape hatches, and the four edge path types with markers / dashes / labels.

In [ ]:
quickstart = Flow(
    nodes=[
        Node("rect",  "Rectangle",     pos=(0,   0),   shape=Shape.rectangle,     fill="#e0f2fe"),
        Node("round", "Rounded",       pos=(200, 0),   shape=Shape.rounded,       fill="#f0fdf4"),
        Node("stad",  "Stadium",       pos=(400, 0),   shape=Shape.stadium,       fill="#fef9c3"),
        Node("ell",   "Ellipse",       pos=(600, 0),   shape=Shape.ellipse,       fill="#fce7f3"),
        Node("dia",   "Diamond",       pos=(0,   160), shape=Shape.diamond,       fill="#ede9fe"),
        Node("para",  "Parallelogram", pos=(200, 160), shape=Shape.parallelogram, fill="#fff7ed"),
        Node("hex",   "Hexagon",       pos=(400, 160), shape=Shape.hexagon,       fill="#f0fdf4"),
        Node("cyl",   "Cylinder",      pos=(600, 160), shape=Shape.cylinder,      fill="#fef2f2"),
        Node("svg",   "Custom SVG",    pos=(0,   320),
             svg_path="M 10,30 A 20,20,0,0,1,50,30 A 20,20,0,0,1,90,30 Q 90,60,50,90 Q 10,60,10,30 z",
             fill="#fce7f3", width=100, height=100),
        Node("html",  "",              pos=(200, 320),
             html="<b style='color:#1d4ed8'>Raw HTML</b><br/><em>node body</em>"),
    ],
    edges=[
        Edge("rect",  "dia",  label="bezier",     path_type="bezier"),
        Edge("round", "para", label="step",       path_type="step",       dash="dashed"),
        Edge("stad",  "hex",  label="smoothstep", path_type="smoothstep", dash="dotted"),
        Edge("ell",   "cyl",  label="straight",   path_type="straight",   marker_start="arrow"),
    ],
    height=600,
)
quickstart

## 02 · Grouping & layout

`Flow.group()` bundles nodes into a labeled container; `Flow.layout()` auto-arranges with dagre. The canvas re-arranges, and the new coordinates are available immediately via `pipeline.positions()`.

In [ ]:
pipeline = Flow(
    nodes=[
        Node("a", "Fetch",    pos=(0,   0),   shape=Shape.rounded,  fill="#e0f2fe"),
        Node("b", "Parse",    pos=(160, 0),   shape=Shape.rounded,  fill="#e0f2fe"),
        Node("c", "Validate", pos=(320, 0),   shape=Shape.rounded,  fill="#e0f2fe"),
        Node("d", "Store",    pos=(0,   160), shape=Shape.cylinder, fill="#f0fdf4"),
        Node("e", "Notify",   pos=(160, 160), shape=Shape.stadium,  fill="#fef9c3"),
    ],
    edges=[Edge("a", "b"), Edge("b", "c"), Edge("c", "d"), Edge("c", "e")],
    height=480,
)
pipeline.group(["a", "b", "c"], label="ETL Pipeline")
pipeline.layout(direction="LR")
pipeline

## 03 · Serialization

Lossless `to_json()` / `from_json()` round-trip and a lossy `to_mermaid()` export.

In [ ]:
flow3 = Flow(
    nodes=[
        Node("start", "Start",   pos=(0,   0), shape=Shape.stadium,   fill="#e8f0fe"),
        Node("proc",  "Process", pos=(0, 120), shape=Shape.rectangle, fill="#ffffff"),
        Node("end",   "End",     pos=(0, 240), shape=Shape.stadium,   fill="#e6f4ea"),
    ],
    edges=[
        Edge("start", "proc", label="next"),
        Edge("proc",  "end",  label="done", dash="dashed"),
    ],
)

snapshot = flow3.to_json()
restored = Flow.from_json(snapshot)
assert restored.to_json() == snapshot   # exact round-trip
print("Round-trip OK:", len(snapshot), "chars")
print(flow3.to_mermaid(direction="TB"))
flow3

## 04 · Custom component (L3 escape hatch)

A custom JS node registered via `register_node_type`, sending click events to Python through `emit` / `Flow.on`. The module reads React from `globalThis.figureflow` so it shares figureflow's vendored React, and is inlined as a `data:` URL so it needs no file server.

`Flow.on` callbacks fire outside any cell's execution context, so a bare `print` in a handler is unreliable. Two robust patterns: **A** append payloads to a list you inspect in a later cell, and **B** route them into an `ipywidgets.Output` to watch live.

In [ ]:
import urllib.parse
import ipywidgets as widgets

MODULE_SRC = """
const { React, xyflow: { Handle, Position } } = globalThis.figureflow;
export default function ClickableNode({ data, selected }) {
  const { emit } = data;                 // emit is injected into data by figureflow
  return React.createElement(
    "div",
    {
      onClick: () => emit("clicked", { id: data.label }),
      style: {
        padding: "10px 16px",
        background: selected ? "#dbeafe" : "#f8fafc",
        border: "2px solid #3b82f6",
        borderRadius: 8,
        cursor: "pointer",
      },
    },
    React.createElement(Handle, { type: "target", position: Position.Top }),
    data.label,
    React.createElement(Handle, { type: "source", position: Position.Bottom })
  );
}
"""

# A data: URL makes the module reachable by the browser without a file server.
MODULE_URL = "data:text/javascript," + urllib.parse.quote(MODULE_SRC)

custom = Flow(
    nodes=[
        Node("x", "Click me!", pos=(0,   0), type="clickable"),
        Node("y", "Or me!",    pos=(200, 0), type="clickable"),
    ],
    edges=[Edge("x", "y")],
    height=300,
)
custom.register_node_type("clickable", MODULE_URL)

# Pattern A: collect payloads into a list, inspect in a later cell.
clicks = []
custom.on("clicked", clicks.append)

# Pattern B: route handler output into an Output widget shown next to the diagram.
out = widgets.Output()

@out.capture()
def on_clicked(payload):
    print("Node clicked:", payload)

custom.on("clicked", on_clicked)

Render the diagram and the live Output side by side, then click the nodes:

In [ ]:
from IPython.display import display

display(custom, out)   # click nodes above; `out` updates live (pattern B)

After clicking, run this cell to see the payloads collected by **pattern A**:

In [ ]:
clicks   # e.g. [{'id': 'Click me!'}, {'id': 'Or me!'}]

## 05 · Display targets

One `Flow`, three transports: `display()` (this notebook), `to_html()` (a self-contained offline snapshot), and `serve()` (a live browser tab). The notebook door is just rendering the widget:

In [ ]:
targets = Flow(
    nodes=[
        Node("start", "Start",   pos=(0,   0), shape=Shape.stadium,   fill="#e8f0fe"),
        Node("proc",  "Process", pos=(0, 120), shape=Shape.rectangle, fill="#ffffff"),
        Node("end",   "End",     pos=(0, 240), shape=Shape.stadium,   fill="#e6f4ea"),
    ],
    edges=[
        Edge("start", "proc", label="next"),
        Edge("proc",  "end",  label="done", dash="dashed"),
    ],
)
targets

The static door writes a self-contained, offline-interactive HTML file:

In [ ]:
html = targets.to_html("diagram.html", title="Display targets")
print(f"Wrote diagram.html ({len(html)} chars) - open it with no network.")

The server door (`targets.serve()`) starts a live, bidirectional browser tab over a dependency-free localhost server. It opens a browser and runs until `targets.stop()`, so it is best driven from a script (`python examples/05_display_targets.py --serve`) rather than a notebook cell.

## 06 · From mermaid (v3)

`Flow.from_mermaid()` parses a pinned flowchart subset into a live Flow: node shapes map from the bracket syntax, edge labels carry over, and a `subgraph … end` block imports as a one-level group. The mermaid text has no coordinates, so every node is *unplaced* and the renderer auto-lays them out on mount via `layout_direction` (set here from the `flowchart TD` header).

In [ ]:
mermaid_flow = Flow.from_mermaid("""flowchart TD
    A([Submit request]) --> B{Reviewer}
    B -->|approve| C[Provision access]
    B -->|reject| D[Send rejection]
    C --> E([Done])
    D --> E
""")

# No coordinates were parsed, so positions() is empty until the renderer
# auto-arranges on mount.
print("placed nodes:", mermaid_flow.positions() or "none - auto-laid on mount")
mermaid_flow

## 07 · LLM ingestion — the repair loop (v3)

`from_json` (and `from_mermaid`, and the MCP tools) route through a single `validate()` funnel that *collects* every fault into one `FlowValidationError` instead of failing on the first. An LLM reads all the lines, fixes them together, and re-emits once.

Given broken `figureflow/1` JSON with three faults — an invalid shape, a duplicate id, and an edge pointing at a missing node — the single error reads:

```
nodes[0].shape: 'circle' is not a valid shape. Use one of: rectangle, rounded, stadium, ellipse, diamond, parallelogram, hexagon, cylinder.
nodes[1].id: duplicate node id 'a'. Node ids must be unique.
edges[0].target: target 'end' names no node. Known node ids: a.
```

The agent renames the duplicate, picks a valid shape (`ellipse`), and adds the missing `end` node; the repaired JSON validates cleanly and renders laid-out. Run it end to end with `python examples/07_llm_ingestion.py`.

## 08 · MCP server (v3)

The `figureflow-mcp` stdio server (behind the `figureflow[mcp]` extra) wraps `serve()`: an agent calls the tools below to build/replace the diagram while a human drags nodes in a live browser tab, and `get_diagram` reads the human's edits back. The server is a *client* of the public surface only — core figureflow stays dependency-free.

| Tool | Does |
|------|------|
| `create_diagram` | Build from JSON or mermaid and serve a browser tab for the human. |
| `get_diagram` | Read the canonical state back, including the human's drag edits. |
| `replace_diagram` | Swap the live diagram wholesale (the running server is untouched). |
| `add_elements` | Add nodes/edges incrementally; unplaced nodes auto-place on render. |
| `close_diagram` | Stop the server and end the session. |

Install with `pip install 'figureflow[mcp]'`, launch with `figureflow-mcp` (an MCP client spawns it over stdin/stdout), and wire it into an agent host, e.g. Claude Desktop's config:

```json
{ "mcpServers": { "figureflow": { "command": "figureflow-mcp" } } }
```

See `examples/08_mcp_server.py` for the same reference as a script.